# CareLoop Clinics: Data Ingestion, Validation & Feature Pipeline
**Objective:** Ingest raw appointment and clinic records, isolate malformed entries for audit, standardize fields, treat statistical outliers via IQR capping, enrich clinic coordinates with live ambient temperature data, and export clean CSVs alongside MySQL `INSERT` scripts.

In [ ]:
import csv
import json
from pathlib import Path
import numpy as np
import pandas as pd
import requests

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

### 1. Schema Audit & Row-Level Quarantine
Before loading into pandas, we run a lightweight type-validation pass on `appointment.csv` so any rows with corrupt numeric fields are logged into `rejected_rows.json` for upstream engineering review.

In [ ]:
NUMERIC_SCHEMA = {
    "age": int,
    "fee_inr": int,
    "prior_no_shows": int,
    "distance_km": float,
}

def validate_record(record: dict) -> tuple[dict | None, str | None, str | None]:
    """Attempts to cast numeric fields according to NUMERIC_SCHEMA."""
    cleaned = record.copy()
    for field, caster in NUMERIC_SCHEMA.items():
        try:
            cleaned[field] = caster(cleaned[field])
        except (ValueError, TypeError) as exc:
            return None, field, str(exc)
    return cleaned, None, None

quarantined_records = []
valid_records = []

with open(DATA_DIR / "appointment.csv", mode="r", newline="", encoding="utf-8") as infile:
    csv_reader = csv.DictReader(infile)
    columns = csv_reader.fieldnames or []
    
    for row in csv_reader:
        parsed_row, bad_col, err_msg = validate_record(row)
        if parsed_row is None:
            quarantined_records.append(row)
        else:
            valid_records.append(parsed_row)

total_ingested = (csv_reader.line_num - 1) if "appointment_id" in columns else csv_reader.line_num
print(f"Detected Columns : {columns}")
print(f"Total Raw Rows   : {total_ingested}")
print(f"Quarantined Rows : {len(quarantined_records)}")

with open(DATA_DIR / "rejected_rows.json", mode="w", encoding="utf-8") as outfile:
    json.dump(quarantined_records, outfile, indent=2)

### 2. Deduplication, Missing Value Treatment & Outlier Capping
Next, we load the dataset into pandas to standardize categorical text (`booking_channel`), parse dates, impute missing numeric values using medians, and apply $1.5 \times \text{IQR}$ winsorization to `distance_km` and `fee_inr`.

In [ ]:
appts = pd.read_csv(DATA_DIR / "appointment.csv")
print(f"Initial shape: {appts.shape}")
print("\nMissing values before cleaning:\n", appts.isnull().sum()[appts.isnull().sum() > 0])

# Remove exact duplicates
dup_count = appts.duplicated().sum()
appts = appts.drop_duplicates().copy()
print(f"\nDropped {dup_count} duplicate rows -> {len(appts)} remaining.")

# Standardize text and dates
appts["booking_channel"] = appts["booking_channel"].astype(str).str.strip().str.title()
appts["appt_date"] = pd.to_datetime(appts["appt_date"], errors="coerce")
appts["booked_on"] = pd.to_datetime(appts["booked_on"], errors="coerce")

# Drop rows missing critical target or appointment date
appts = appts.dropna(subset=["appt_date", "status"]).copy()
print(f"After dropping invalid dates/status -> {len(appts)} rows.")

# Sanitize age (valid clinical range: 1 to 100) and impute numeric medians
appts["age"] = pd.to_numeric(appts["age"], errors="coerce")
appts.loc[(appts["age"] < 1) | (appts["age"] > 100), "age"] = np.nan

age_median = appts["age"].median()
dist_median = appts["distance_km"].median()

appts["age"] = appts["age"].fillna(age_median).round().astype(int)
appts["distance_km"] = appts["distance_km"].fillna(dist_median)

# Winsorize continuous features using 1.5 * IQR fences
for feature in ["distance_km", "fee_inr"]:
    q25, q75 = appts[feature].quantile([0.25, 0.75])
    iqr = q75 - q25
    lower_bound = q25 - 1.5 * iqr
    upper_bound = q75 + 1.5 * iqr
    
    outliers_clipped = ((appts[feature] < lower_bound) | (appts[feature] > upper_bound)).sum()
    appts[feature] = np.clip(appts[feature], lower_bound, upper_bound)
    print(f"[{feature}] Bounds: ({lower_bound:.2f}, {upper_bound:.2f}) | Clipped: {outliers_clipped} values")

appts["distance_km"] = appts["distance_km"].round(1)
appts["fee_inr"] = appts["fee_inr"].round().astype(int)

### 3. Feature Engineering & NumPy Statistical Profiling
We derive `lead_days` (days elapsed between booking and appointment) and encode our binary target `no_show` (`1` = No-Show, `0` = Attended). We also run vectorized NumPy checks to examine lead-time tails and distance dispersion.

In [ ]:
appts["lead_days"] = (appts["appt_date"] - appts["booked_on"]).dt.days
appts["no_show"] = (appts["status"] == "No-Show").astype(int)

# Vectorized summary diagnostics
numeric_matrix = appts[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()
print("Feature Means (Age, Dist, Fee, Lead Days):", np.round(np.mean(numeric_matrix, axis=0), 2))

long_lead_mask = appts["lead_days"].to_numpy() > 14
print(f"Bookings > 14 days lead time: {long_lead_mask.sum()} ({long_lead_mask.mean():.2%})")

distances = appts["distance_km"].to_numpy()
distance_z = (distances - distances.mean()) / distances.std()
print(f"Distance observations with |z| > 2: {np.sum(np.abs(distance_z) > 2)}")

### 4. Clinic Geolocation Weather Enrichment & SQL Export
We query the Open-Meteo Forecast API for each clinic's coordinates (with a retry fallback) to attach current ambient temperatures, run final data integrity assertions, and generate `appointments_clean.csv`, `clinics_enriched.csv`, and `load_data.sql`.

In [ ]:
clinics = pd.read_csv(DATA_DIR / "clinics.csv")

def fetch_clinic_temp(lat: float, lon: float, clinic_name: str, retries: int = 2) -> float | None:
    endpoint = "https://api.open-meteo.com/v1/forecast"
    query_params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": "temperature_2m",
        "forecast_days": 1,
    }
    for attempt in range(retries):
        try:
            resp = requests.get(endpoint, params=query_params, timeout=10)
            resp.raise_for_status()
            payload = resp.json()
            return payload.get("hourly", {}).get("temperature_2m", [None])[0]
        except requests.RequestException as exc:
            if attempt == retries - 1:
                print(f"Warning: Weather lookup failed for {clinic_name} ({exc})")
    return None

clinics["temp_c"] = [
    fetch_clinic_temp(r["latitude"], r["longitude"], r["clinic_name"])
    for _, r in clinics.iterrows()
]
clinics.to_csv("clinics_enriched.csv", index=False)

# Final formatting and schema ordering
appts = appts.sort_values("appointment_id").copy()
appts["booked_on"] = appts["booked_on"].dt.strftime("%Y-%m-%d")
appts["appt_date"] = appts["appt_date"].dt.strftime("%Y-%m-%d")

ORDERED_COLS = [
    "appointment_id", "patient_id", "clinic_id", "booked_on", "appt_date",
    "age", "gender", "distance_km", "fee_inr", "booking_channel",
    "visit_type", "reminder_sent", "prior_no_shows", "status",
    "lead_days", "no_show"
]
appts = appts[ORDERED_COLS]

# Integrity checks
assert appts["appointment_id"].is_unique, "Duplicate appointment_id detected!"
assert not appts.isnull().any().any(), "Null values remain in cleaned dataframe!"

appts.to_csv("appointments_clean.csv", index=False)

# Generate deterministic SQL seed script
with open("load_data.sql", mode="w", encoding="utf-8") as sql_file:
    for _, r in clinics.iterrows():
        temp_sql = "NULL" if pd.isna(r["temp_c"]) else r["temp_c"]
        sql_file.write(
            f"INSERT INTO clinics VALUES ('{r['clinic_id']}','{r['clinic_name']}','{r['city']}','{r['tier']}',{r['latitude']},{r['longitude']},{temp_sql});\n"
        )
    for _, r in appts.iterrows():
        sql_file.write(
            f"INSERT INTO appointments VALUES ('{r['appointment_id']}','{r['patient_id']}','{r['clinic_id']}','{r['booked_on']}','{r['appt_date']}',{r['age']},'{r['gender']}',{r['distance_km']},{r['fee_inr']},'{r['booking_channel']}','{r['visit_type']}','{r['reminder_sent']}',{r['prior_no_shows']},'{r['status']}',{r['lead_days']},{r['no_show']});\n"
        )

print("Pipeline complete: exported clinics_enriched.csv, appointments_clean.csv, and load_data.sql.")